# Benchmark precomputed GPN-Star scores on TraitGym

This workflow annotates the real [`songlab/ukb_finemapped_nc_traitgym`](https://huggingface.co/datasets/songlab/ukb_finemapped_nc_traitgym) benchmark from the public [`songlab/gpn-star-scores`](https://huggingface.co/datasets/songlab/gpn-star-scores) Parquet shards, then computes area under the precision-recall curve (AUPRC). It does not download a model or whole-genome alignment.

The join keys are `chrom`, one-based `pos`, `ref`, and `alt`. They must use the same reference assembly and chromosome spelling as the selected score set.

In [ ]:
%pip install -q "polars[rtcompat]>=1.42,<2" "huggingface-hub>=1.4,<2" "scikit-learn>=1.8,<2"

## TraitGym benchmark

This matched benchmark contains 1,140 fine-mapped UK Biobank non-coding variants and 10,260 controls. We pin the exact public dataset revision.

In [ ]:
import polars as pl
from sklearn.metrics import average_precision_score

KEYS = ["chrom", "pos", "ref", "alt"]
TRAITGYM_REVISION = "947e2320f66b83489063aa41252ff76498aeefbb"
TRAITGYM_ROOT = (
    "hf://datasets/songlab/ukb_finemapped_nc_traitgym@"
    f"{TRAITGYM_REVISION}"
)
variants = pl.read_parquet(f"{TRAITGYM_ROOT}/test.parquet")
variants.select(KEYS + ["label", "match_group"]).head()

## Fast reference metric

TraitGym mirrors the released GPN-Star-M447 prediction column in a small file. This gives a fast metric check before demonstrating the general genome-wide lookup below. Its `score` has the same LLR direction, so we negate it: higher `effect_score` should rank positives first.

In [ ]:
cached_scores = pl.read_parquet(
    f"{TRAITGYM_ROOT}/predictions/GPN-Star-M447.parquet"
).get_column("score")
cached_auprc = average_precision_score(
    variants.get_column("label").to_numpy(),
    -cached_scores.to_numpy(),
)
print(f"GPN-Star-M447 TraitGym AUPRC: {cached_auprc:.4f}")

## Read only the required shard

The released tables are partitioned by chromosome. Filtering positions before each join avoids downloading all genome-wide scores, although this full 22-autosome remote benchmark can still take several minutes. For large or repeated analyses, download the required shards once and point `SCORE_ROOT` to the local directory.

In [ ]:
DATASET_REVISION = "5c799b2ec6aa089f0caa8294ae72adb4510f81ae"
SCORE_SET = "gpn-star-hg38-m447-200m"
SCORE_ROOT = (
    "hf://datasets/songlab/gpn-star-scores@"
    f"{DATASET_REVISION}/data/{SCORE_SET}/llr"
)

results = []
for chrom in variants.get_column("chrom").unique(maintain_order=True):
    chrom_variants = variants.filter(pl.col("chrom") == chrom)
    positions = chrom_variants.get_column("pos").unique().to_list()
    scores = (
        pl.scan_parquet(f"{SCORE_ROOT}/llr_chr{chrom}.parquet")
        .filter(pl.col("pos").is_in(positions))
    )
    results.append(
        chrom_variants.lazy()
        .join(scores, on=KEYS, how="left")
        .collect(engine="streaming")
    )

annotated = pl.concat(results)
missing = annotated.get_column("llr_calibrated").null_count()
assert missing == 0, f"{missing} variants were absent from the score table"
annotated

`llr_calibrated` is the mutation-rate-calibrated alternate-versus-reference log-likelihood ratio; more-negative values indicate greater constraint or predicted effect. Therefore the classifier ranking score is `-llr_calibrated`. `abs_llr_calibrated` is an independently calibrated score and must not be recomputed as `abs(llr_calibrated)`.

In [ ]:
annotated = annotated.with_columns(
    (-pl.col("llr_calibrated")).alias("effect_score")
)
auprc = average_precision_score(
    annotated.get_column("label").to_numpy(),
    annotated.get_column("effect_score").to_numpy(),
)
print(f"Genome-wide join AUPRC: {auprc:.4f}")
print(f"Cached prediction AUPRC: {cached_auprc:.4f}")

In [ ]:
annotated.write_parquet("gpn_star_scored_variants.parquet")